# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 4cef8b3c-6522-4394-856d-2638ec4d4750
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 4cef8b3c-6522-4394-856d-2638ec4d4750 to get into ready status...
Session 4cef8b3c-6522-4394-856d-2638ec4d4750 

#### Ler os dados da camada Bronze

In [ ]:
path_bronze = "s3://datalake-404604958697/bronze/pnad_covid/"

df_bronze = spark.read.parquet(path_bronze)

df_bronze.show(5)
df_bronze.printSchema()

+----+---+-------+-------+-----+-----+-----+-----+-------+---------+-----+-----+------+------------+------------+------+----+-----+------+------+------+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+------+------+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+----+----+----+----+----+----+----+-----+-----+-----+----+----+-----+-----+-----+-----+-----+------+------+----+----+----+-----+------+------+-----+------+------+-----+-----+-----+------+-------+-------+------+-------+-------+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+------+-----+------+-----+-----+-----+-----+----+-----+-----+-----+----+---+
| ano| UF|CAPITAL|RM_RIDE|V1008|V1012|V1013|V1016|Estrato|      UPA|V1022|V1023| V1030|       V1031|       V1032|posest|A001|A001A|A001B1|A001B2|A001B3|A002|A003|A004|A005|B0011|B0012|B0013|B0014|B0015|B0016|B0017|B0018|B0019|B00110|B00111|B00112|B002|B0031|B0032|B0033|B0034|B0035|B0036|B

#### Selecionando colunas que serão métricas



In [ ]:
colunas = [
    "ano","mes",
    #Caracteristicas
    "A002", #Idade
    "A003", #Sexo
    "V1022", # Urbano x Rural
    "UF","CAPITAL",
    "A001A", # Parentesco
    #Sintomas
    "B0011", #Febre
    "B0012", #Tosse
    "B0014", #Respiracao
    "B0015", #Dor de Cabeça
    "B00111", #Paladar
    #Comportamento
    "B0043", #buscou_atendimento_sus
    "B0045", #buscou_atendimento_privado
    "B005", #ficou_internado
    "B006", #ficou_entubado
    "B0031", #ficou_em_casa
    #Economicas
    "C002", #se_afastou
    "C003", #motivo_afastamento
    "C007C", #cargo
    "C007D", #ramo
    "C01011" #faixa_renda
]

df_silver = df_bronze.select(*colunas)

In [ ]:
df_silver.show(10)
df_silver.printSchema()

+----+---+----+----+-----+---+-------+-----+-----+-----+-----+-----+------+-----+-----+----+----+-----+----+----+-----+-----+------+
| ano|mes|A002|A003|V1022| UF|CAPITAL|A001A|B0011|B0012|B0014|B0015|B00111|B0043|B0045|B005|B006|B0031|C002|C003|C007C|C007D|C01011|
+----+---+----+----+-----+---+-------+-----+-----+-----+-----+-----+------+-----+-----+----+----+-----+----+----+-----+-----+------+
|2020| 10|  36|   1|    1| 11|   11.0|    1|    2|    2|    2|    2|     2| NULL| NULL|NULL|NULL| NULL|NULL|NULL| 35.0|  6.0|   4.0|
|2020| 10|  30|   2|    1| 11|   11.0|    2|    2|    2|    2|    2|     2| NULL| NULL|NULL|NULL| NULL|NULL|NULL| 27.0| 20.0|   4.0|
|2020| 10|  13|   1|    1| 11|   11.0|    4|    2|    2|    2|    2|     2| NULL| NULL|NULL|NULL| NULL|NULL|NULL| NULL| NULL|  NULL|
|2020| 10|  11|   1|    1| 11|   11.0|    4|    2|    2|    2|    2|     2| NULL| NULL|NULL|NULL| NULL|NULL|NULL| NULL| NULL|  NULL|
|2020| 10|  57|   2|    1| 11|   11.0|    1|    2|    2|    2|    2| 

In [ ]:
from pyspark.sql.functions import count

df_count = df_silver.groupBy("ano","mes") \
    .agg(count("*").alias("qtd_linhas")) \
    .orderBy("ano","mes")

df_count.show(50, False)

+----+---+----------+
|ano |mes|qtd_linhas|
+----+---+----------+
|2020|5  |349306    |
|2020|6  |381270    |
|2020|7  |384166    |
|2020|8  |386520    |
|2020|9  |387298    |
|2020|10 |380461    |
|2020|11 |381438    |
+----+---+----------+


#### Renomeando colunas


In [ ]:
df_silver = df_silver.withColumnRenamed("A002", "idade") \
                     .withColumnRenamed("A003", "sexo") \
                     .withColumnRenamed("V1022", "urbano_rural") \
                     .withColumnRenamed("UF", "uf") \
                     .withColumnRenamed("CAPITAL", "capital") \
                     .withColumnRenamed("A001A", "parentesco") \
                     .withColumnRenamed("B0011", "febre") \
                     .withColumnRenamed("B0012", "tosse") \
                     .withColumnRenamed("B0014", "dificuldade_respiratoria") \
                     .withColumnRenamed("B0015", "dor_de_cabeca") \
                     .withColumnRenamed("B00111", "perda_olfato_paladar") \
                     .withColumnRenamed("B0043", "buscou_atendimento_sus") \
                     .withColumnRenamed("B0045", "buscou_atendimento_privado") \
                     .withColumnRenamed("B005", "ficou_internado") \
                     .withColumnRenamed("B006", "ficou_entubado") \
                     .withColumnRenamed("B0031", "ficou_em_casa") \
                     .withColumnRenamed("C002", "se_afastou") \
                     .withColumnRenamed("C003", "motivo_afastamento") \
                     .withColumnRenamed("C007C", "cargo") \
                     .withColumnRenamed("C007D", "ramo") \
                     .withColumnRenamed("C01011", "faixa_renda")


#### Conversões

In [ ]:
from pyspark.sql.functions import col, lpad
df_silver = df_silver \
    .withColumn("uf",col("uf").cast("int")) \
    .withColumn("capital",col("capital").cast("int")) \
    .withColumn("buscou_atendimento_sus",col("buscou_atendimento_sus").cast("int")) \
    .withColumn("buscou_atendimento_privado",col("buscou_atendimento_privado").cast("int")) \
    .withColumn("ficou_internado",col("ficou_internado").cast("int")) \
    .withColumn("ficou_entubado",col("ficou_entubado").cast("int")) \
    .withColumn("ficou_em_casa",col("ficou_em_casa").cast("int")) \
    .withColumn("se_afastou",col("se_afastou").cast("int")) \
    .withColumn("motivo_afastamento",col("motivo_afastamento").cast("int")) \
    .withColumn("cargo",col("cargo").cast("int")) \
    .withColumn("ramo",col("ramo").cast("int")) \
    .withColumn("faixa_renda",col("faixa_renda").cast("int"))

colunas_string_codigo = ["faixa_renda"]
for c in colunas_string_codigo:
    df_silver = df_silver.withColumn(
        c,
        lpad(col(c).cast("string"), 2, "0")
    )

In [ ]:
df_silver.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- idade: long (nullable = true)
 |-- sexo: long (nullable = true)
 |-- urbano_rural: long (nullable = true)
 |-- uf: integer (nullable = true)
 |-- capital: integer (nullable = true)
 |-- parentesco: long (nullable = true)
 |-- febre: long (nullable = true)
 |-- tosse: long (nullable = true)
 |-- dificuldade_respiratoria: long (nullable = true)
 |-- dor_de_cabeca: long (nullable = true)
 |-- perda_olfato_paladar: long (nullable = true)
 |-- buscou_atendimento_sus: integer (nullable = true)
 |-- buscou_atendimento_privado: integer (nullable = true)
 |-- ficou_internado: integer (nullable = true)
 |-- ficou_entubado: integer (nullable = true)
 |-- ficou_em_casa: integer (nullable = true)
 |-- se_afastou: integer (nullable = true)
 |-- motivo_afastamento: integer (nullable = true)
 |-- cargo: integer (nullable = true)
 |-- ramo: integer (nullable = true)
 |-- faixa_renda: string (nullable = true)


In [ ]:
from pyspark.sql.functions import count

df_silver.groupBy("ano","mes") \
    .agg(count("*").alias("qtd_linhas")) \
    .orderBy("ano","mes") \
    .show(50, False)

+----+---+----------+
|ano |mes|qtd_linhas|
+----+---+----------+
|2020|5  |349306    |
|2020|6  |381270    |
|2020|7  |384166    |
|2020|8  |386520    |
|2020|9  |387298    |
|2020|10 |380461    |
|2020|11 |381438    |
+----+---+----------+


In [ ]:
df_silver.select("faixa_renda").distinct().show()

+-----------+
|faixa_renda|
+-----------+
|       NULL|
|         01|
|         07|
|         08|
|         09|
|         06|
|         05|
|         03|
|         04|
|         00|
|         02|
+-----------+


#### Salvando Tabela

In [ ]:
path_silver = "s3://datalake-404604958697/silver/pnad_covid/base_tratada/"

df_silver.write \
    .mode("overwrite") \
    .partitionBy("ano","mes") \
    .format("parquet") \
    .save(path_silver)